# Phutball: gated GRPO v2 (curve gate, targeting v2)

The recipe of `grpo_gated_colab.ipynb` (cost-sensitive GRPO from v4 r14, `--force-ok 0 --cut-cost 0.35`, 20 steps of
4 puzzles x 8 samples per phase) with three changes, all from what the first run showed:

1. **Curve gate** (`curriculum_pp gate --curve`, gate v4): validation is ONE rollout per item at 16k, scored at 2k / 4k /
   8k / 16k (`llm_bench --salvage --truncate-at`). Promote unless, paired against the best and the peak, (a) the curve
   score (mean over budgets of the finished answer's utility: +1 / -1 / -1.5 / 0 not yet) is clearly worse, (b) the
   peak (forced accuracy at 16k) is clearly lower, or (c) one family's forced accuracy at 16k collapses. The 6k gate
   rewarded finishing within one budget only: on the transfer bridge every model was cut off on 92-100% of the hardest
   puzzles at 6k, and the untrained model reached 87% at 24k.
2. **Headroom** (`grpo_targets --headroom 0.5`): bucket weight = gap + 0.5 x 4q(1-q); prevent / forced placements are
   no longer starved.
3. **Base-rate correction** (`grpo_pilot --prior-correct`): rewards weighted by validation share / draw share, so the
   targeted draw does not teach the wrong base rates (run 1, p13 -> p14: a 29% no-win draw, forced false NO WIN 48 -> 65).

4. **Curve reward** (`grpo_pilot --reward curve`, `REWARD = 'curve'`): the gate's objective as the reward, i.e. the
   expected cost reward under an unknown deadline drawn log-uniformly from 2k-16k, computed from one rollout: a
   finished answer's utility x w(t), w = 1 / 0.75 / 0.5 / 0.25 at 2k / 4k / 8k / 16k (smooth in between); cut off at 16k:
   a wrong forced answer x 0.25, minus 0.1. Rollouts run to 16k, so hard puzzles solved past 6k earn credit (run 1 gave
   them none), the same answer earlier earns more at every scale, and a wrong early answer costs the most.
   `REWARD = 'cost'` keeps run 1's 6k reward.

`START = 'base'` (default) trains a fresh LoRA on untrained Qwen: the surface ladder showed base matching run 1's
best at 24k on the board (forced 91% vs 88%) while expert iteration's r14 had fallen to 66%, so starting from r14 would
mostly re-measure recovery from that damage. `'r14'` / `'p13'` start from v4 r14 / run 1's p13. Separate folder (`grpo_v2_<START>_<REWARD>`): run 1 is untouched.
**Runtime:** A100, High-RAM; each phase ~1.5x run 1 (the 16k validation eval). Resumable: re-run all cells.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request, gzip, shutil
V4 = '/content/drive/MyDrive/phutball/curriculum_v4'; RUN1 = '/content/drive/MyDrive/phutball/grpo_cost'
START = 'base'                                  # 'base': fresh LoRA on untrained Qwen (the ladder: base reaches run 1's
                                                #   best at 24k, and the SFT stages lowered the ceiling: r14 66% vs base
                                                #   91% forced on the board); 'r14': from v4 r14; 'p13': run 1's p13
INIT = {'base': None, 'r14': f'{V4}/r14/adapter/final', 'p13': f'{RUN1}/p13/final'}[START]
POOL = 'llm_train/puzzles/pool_v4.jsonl.gz'; FMT = 'ascii+men'
TEST = 'llm_bench/data/bench_v2.jsonl'; VAL = 'llm_train/puzzles/val_v3.jsonl'
REWARD = 'curve'                                # 'curve': the curve gate's objective as the reward (rollouts to 16k);
                                                #  'cost': run 1's reward (6k cap, per-token lam, cut cost 0.35)
CAP = 16384 if REWARD == 'curve' else 6144      # training rollouts (the curve reward's largest deadline)
OUT = f'/content/drive/MyDrive/phutball/grpo_v2_{START}_{REWARD}'; os.makedirs(OUT, exist_ok=True)
STATE = f'{OUT}/gate_state.json'; PUZ = f'{OUT}/grpo_puzzles.jsonl'
EVAL_CAP = 16384; CURVE = '2048,4096,8192,16384'   # validation: one rollout at EVAL_CAP, scored at every budget
TARGET_AT = CAP                                  # targeting measures the gap at the training cap (an extra cut, not gated)
AT_FLAG = '' if TARGET_AT == EVAL_CAP else f'--at {TARGET_AT}'
STEPS = 20; PHASES = 16; PG = 25
TARGET_FLAGS = '--headroom 0.5'                 # run 1: --headroom 0
PILOT_FLAGS = '--prior-correct'                 # run 1: --no-prior-correct
COST = {'curve': '--cost-a 1.0 --cost-b 1.5 --cost-c 0.25 --curve-min 2048 --cut-cost 0.1',
        'cost': '--cost-a 1.0 --cost-b 1.5 --cost-c 0.25 --lam 3e-5 --lam-ramp 50 --force-ok 0.0 --cut-cost 0.35'}[REWARD]
print('start:', INIT or 'untrained Qwen3.5-9B (fresh LoRA)', '->', OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=EVAL_CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
CUTS = [int(c) for c in CURVE.split(',')]
EXTRA = [TARGET_AT] if TARGET_AT not in CUTS else []             # cuts for targeting only
def curve_files(name):
    return [f'{name}.jsonl', f'{name}.summary.json', f'{name}-forced.summary.json'] + \
           [f for c in CUTS[:-1] + EXTRA for f in (f'{name}-at{c}.jsonl', f'{name}-at{c}-forced.summary.json')]
def evaluate_curve(name, model, bench=VAL, pg=0):
    """One rollout per item at EVAL_CAP, forced at the cap, then cut and forced at every smaller budget."""
    if all(os.path.exists(f'{EV}/{f}') for f in curve_files(name)): print('done:', name); return
    !python -m llm_bench.run --bench {bench} --model {model} --name {name} --prompt-format {FMT} --thinking true --qwen-thinking-sampling --samples 1 --max-tokens {EVAL_CAP} --per-group {pg} --concurrency 96 --timeout 3600 | tail -14
    !python -m llm_bench.run --bench {bench} --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --per-group {pg} --concurrency 48 --timeout 3600 | tail -14
    for c in CUTS[:-1] + EXTRA:
        !python -m llm_bench.run --bench {bench} --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --truncate-at {c} --per-group {pg} --concurrency 48 --timeout 3600 | tail -3
    for f in curve_files(name): shutil.copy(f'llm_bench/results/{f}', f'{EV}/{f}')
def best_adapter():
    return (json.load(open(STATE)).get('best') if os.path.exists(STATE) else None) or INIT
def init_flag():                                # no --init-adapter: grpo_pilot starts a fresh LoRA on the base model
    b = best_adapter(); return f'--init-adapter {b}' if b else ''
def serve_start():                              # (server, model name) of the starting model
    return (serve({'init': INIT}), 'init') if INIT else (serve({}), 'Qwen/Qwen3.5-9B')


## 1. Start: the starting model's validation curve (the first best and peak); the GRPO puzzles

Run 1's 6k evaluations cannot be reused (the curve gate needs one rollout at 16k), so the start is evaluated here.


In [ ]:
if not all(os.path.exists(f'{EV}/{f}') for f in curve_files('v0')):
    server, m0 = serve_start()
    evaluate_curve('v0', m0)
    !pkill -f "vllm serve"
if not os.path.exists(STATE):
    json.dump({"puzzles": {}, "rates": {}, "applied": [], "best": INIT, "best_eval": f'{EV}/v0', "peak_eval": f'{EV}/v0',
               "log": []}, open(STATE, 'w'))
if not os.path.exists(PUZ):
    st4 = json.load(open(f'{V4}/state.json')); active = {p for p, s in st4['puzzles'].items() if s['status'] == 'active'}
    with open(PUZ, 'w') as f:
        for l in gzip.open(POOL, 'rt'):
            if json.loads(l)['id'] in active: f.write(l)
from collections import Counter
print(sum(1 for _ in open(PUZ)), 'puzzles:', dict(Counter(json.loads(l)['subtask'] for l in open(PUZ))))
print('best:', best_adapter())


## 2. Gated GRPO phases (unattended)

In [ ]:
for P in range(1, PHASES + 1):
    d = f'{OUT}/p{P}'
    if not os.path.exists(f'{d}/final/adapter_model.safetensors'):
        print(f'=== phase {P}: GRPO from', best_adapter() or 'untrained Qwen3.5-9B (fresh LoRA)')
        os.makedirs(d, exist_ok=True); puz = f'{d}/puzzles.jsonl'
        if not os.path.exists(puz):
            best_eval = json.load(open(STATE))['best_eval']      # the gap at the training cap (TARGET_AT)
            !python -m llm_train.grpo_targets --puzzles {PUZ} --best-eval {best_eval} --rollouts "{OUT}/p*/rollouts.jsonl" --seed {P} {TARGET_FLAGS} {AT_FLAG} --out {puz}
        !python -m llm_train.grpo_pilot --out {d} {init_flag()} --puzzles {puz} --prompt-format {FMT} --reward {REWARD} {COST} {PILOT_FLAGS} --cap {CAP} --max-steps {STEPS} --hours 99 --save-every 5 --seed {P} 2>&1 | grep --line-buffered -E "puzzles|loaded adapter|reward call|cost reward|saved|rror|Traceback"
    if not all(os.path.exists(f'{EV}/{f}') for f in curve_files(f'v{P}')):
        server = serve({f'p{P}': f'{d}/final'})
        evaluate_curve(f'v{P}', f'p{P}')
        !pkill -f "vllm serve"
    if not os.path.exists(f'{d}/gate.txt'):
        !python -m llm_train.curriculum_pp gate --state {STATE} --round {P} --new {EV}/v{P} --adapter {d}/final --curve {CURVE} | tee {d}/gate.txt
print('done; best:', best_adapter())


## 3. Results, and the held-out test set

Every phase was chosen on the same validation items, so the final best's validation curve flatters it a little; the test
set (`bench_v2`, never used by the gate) scores the start and the final best once, with the same curve evaluation.


In [ ]:
st = json.load(open(STATE))
for e in st['log']:
    if e['event'] == 'gate': print(f"phase {e['round']}: {'PROMOTED' if e['promoted'] else 'kept best'} ({e['reason']})")
print('best:', st['best'])
# held-out test set, once, after the last phase:
# server = serve({**({'init': INIT} if INIT else {}), 'best': best_adapter()})
# evaluate_curve('test-init', 'init' if INIT else 'Qwen/Qwen3.5-9B', bench=TEST, pg=PG)
# evaluate_curve('test-best', 'best', bench=TEST, pg=PG)
# !pkill -f "vllm serve"


In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()